# Minimal PVD on Synthetic Peptides
This notebook reuses PVD training components and swaps ShapeNet for `SyntheticPeptidesDataset`.

In [1]:
import os
import sys
import importlib.util
import subprocess
from types import SimpleNamespace

import torch
from torch.utils.data import DataLoader, random_split

os.environ['CUDA_VISIBLE_DEVICES'] = '2'
# train_generation imports utils.visualize -> trimesh
if importlib.util.find_spec('trimesh') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'trimesh'])

sys.path.append(os.path.abspath('PVD'))

from train_generation import Model, get_betas
from synthetic_peptides_dataset import SyntheticPeptidesDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

Using /home/go73dov/.cache/torch_extensions/py310_cu120 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/go73dov/.cache/torch_extensions/py310_cu120/pvcnn_backend/build.ninja...
/home/go73dov/miniforge3/envs/venv/lib/python3.10/site-packages/torch/utils/cpp_extension.py:1965: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module pvcnn_backend...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)
Loading extension module pvcnn_backend...


ninja: no work to do.
cuda


In [2]:
cfg = SimpleNamespace(
    nc=3,
    npoints=3000,
    bs=8,
    workers=2,
    niter=1000,
    lr=1e-5,
    beta1=0.5,
    decay=0.0,
    attention=False,
    dropout=0.1,
    embed_dim=128,
    voxel_resolution_multiplier=1,
    loss_type='mse',
    model_mean_type='eps',
    model_var_type='fixedsmall',
    schedule_type='linear',
    beta_start=1e-4,
    beta_end=0.02,
    time_num=1000,
    structure_types=None,
)

# Actual training setup: use separate train/val folders (no synthetic overfit split)
train_ds = SyntheticPeptidesDataset(
    data_path='./data/synthetic_peptides_split/train/',
    num_files=300,
    target_num_points=cfg.npoints,
    normalize=True,
    return_peptide_ids=False,
    structure_types=cfg.structure_types,
    seed=42,
)

val_ds = SyntheticPeptidesDataset(
    data_path='./data/synthetic_peptides_split/val/',
    num_files=64,
    target_num_points=cfg.npoints,
    normalize=True,
    return_peptide_ids=False,
    structure_types=cfg.structure_types,
    seed=42,
 )

print('train size:', len(train_ds), '| val size:', len(val_ds))

Loading centroid dataset from: ./data/synthetic_peptides_split/train/
Structure types: lamellar_sheets, micelles, nanofibers, nanotubes, random_aggregates
Selected files per structure:
  lamellar_sheets: 21/700
  micelles: 21/700
  nanofibers: 21/700
  nanotubes: 21/700
  random_aggregates: 214/7000
Dataset ready: 298 files, target_points=3000, normalize=True
Loading centroid dataset from: ./data/synthetic_peptides_split/val/
Structure types: lamellar_sheets, micelles, nanofibers, nanotubes, random_aggregates
Selected files per structure:
  lamellar_sheets: 4/150
  micelles: 4/150
  nanofibers: 4/150
  nanotubes: 4/150
  random_aggregates: 45/1500
Dataset ready: 61 files, target_points=3000, normalize=True
train size: 298 | val size: 61


In [3]:
class PVDAdapter(torch.utils.data.Dataset):
    def __init__(self, subset):
        self.subset = subset

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, i):
        item = self.subset[i]
        return {'idx': i, 'train_points': item['points']}


train_loader = DataLoader(
    PVDAdapter(train_ds),
    batch_size=cfg.bs,
    shuffle=len(train_ds) > 0,
    num_workers=cfg.workers,
    drop_last=False,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    PVDAdapter(val_ds),
    batch_size=cfg.bs,
    shuffle=False,
    num_workers=cfg.workers,
    drop_last=False,
    pin_memory=torch.cuda.is_available(),
)

len(train_loader), len(val_loader)

(38, 8)

In [ ]:
import json
from pathlib import Path

if torch.cuda.is_available():
    torch.cuda.empty_cache()

RUN_NAME = 'pvd_peptides_300'
SAVE_EVERY = 10
VAL_INTERVAL = 10

betas = get_betas(cfg.schedule_type, cfg.beta_start, cfg.beta_end, cfg.time_num)
model = Model(cfg, betas, cfg.loss_type, cfg.model_mean_type, cfg.model_var_type).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.decay, betas=(cfg.beta1, 0.999))

checkpoint_dir = Path('checkpoints') / RUN_NAME
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / 'latest.pth'
best_checkpoint_path = checkpoint_dir / 'best.pth'
log_path = checkpoint_dir / 'training_log.json'


config = {
    'run_name': RUN_NAME,
    'train_data_path': './data/synthetic_peptides_split/train/',
    'val_data_path': './data/synthetic_peptides_split/val/',
    'structure_types': cfg.structure_types,
    'npoints': cfg.npoints,
    'batch_size': cfg.bs,
    'workers': cfg.workers,
    'learning_rate': cfg.lr,
    'epochs': cfg.niter,
    'save_every': SAVE_EVERY,
    'val_interval': VAL_INTERVAL,
    'model_params': sum(p.numel() for p in model.parameters()),
    'loss_type': cfg.loss_type,
    'model_mean_type': cfg.model_mean_type,
    'model_var_type': cfg.model_var_type,
    'schedule_type': cfg.schedule_type,
    'beta_start': cfg.beta_start,
    'beta_end': cfg.beta_end,
    'time_num': cfg.time_num,
    'attention': cfg.attention,
    'embed_dim': cfg.embed_dim,
}

start_epoch = 0
train_losses = []
val_losses = []
best_val = float('inf')
noises_init = torch.randn(len(train_ds), cfg.npoints, cfg.nc, device=device)

if checkpoint_path.exists():
    print('\nLoading existing checkpoint...')
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    train_losses = checkpoint.get('train_losses', [])
    val_losses = checkpoint.get('val_losses', [])
    best_val = checkpoint.get('best_val', min(val_losses) if val_losses else float('inf'))
    if 'noises_init' in checkpoint:
        noises_init = checkpoint['noises_init'].to(device)
    print(f"Resumed from epoch {start_epoch} | best val: {best_val:.6f}")
else:
    print('\nStarting fresh training run')

def save_state(epoch, path):
    state = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_losses': train_losses,
        'val_losses': val_losses,
        'best_val': best_val,
        'config': config,
        'noises_init': noises_init.detach().cpu(),
    }
    torch.save(state, path)
    return state

epoch = start_epoch - 1
try:
    for epoch in range(start_epoch, cfg.niter):
        model.train()
        running = 0.0
        for batch in train_loader:
            x = batch['train_points'].transpose(1, 2).to(device)
            idx = batch['idx'].long().to(device)
            noise_batch = noises_init[idx].transpose(1, 2)

            loss = model.get_loss_iter(x, noise_batch).mean()
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running += loss.item()

        train_loss = running / max(1, len(train_loader))
        train_losses.append(train_loss)

        val_loss = None
        if epoch % VAL_INTERVAL == 0:
            model.eval()
            val_running = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    x = batch['train_points'].transpose(1, 2).to(device)
                    loss_v = model.get_loss_iter(x).mean()
                    val_running += loss_v.item()
            val_loss = val_running / max(1, len(val_loader))
            val_losses.append(val_loss)

            if val_loss < best_val:
                best_val = val_loss
                save_state(epoch, best_checkpoint_path)

        if val_loss is not None:
            print(f"Epoch {epoch:4d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | Best: {best_val:.6f}")
        else:
            print(f"Epoch {epoch:4d} | Train Loss: {train_loss:.6f}")

        if (epoch + 1) % SAVE_EVERY == 0 or epoch == cfg.niter - 1:
            epoch_folder = checkpoint_dir / f'checkpoint_epoch_{epoch + 1}'
            epoch_folder.mkdir(exist_ok=True)
            state = save_state(epoch, checkpoint_path)
            torch.save(state, epoch_folder / 'model.pth')

            payload = {
                'epoch': epoch,
                'train_losses': train_losses,
                'val_losses': val_losses,
                'config': config,
                'best_val': best_val,
            }
            with open(epoch_folder / 'training_log.json', 'w') as f:
                json.dump(payload, f, indent=2)
            with open(log_path, 'w') as f:
                json.dump(payload, f, indent=2)

    final_val_str = f", Final Val Loss: {val_losses[-1]:.6f}" if val_losses else ''
    print(f"\nTraining complete. Final Train Loss: {train_losses[-1]:.6f}{final_val_str}, Best Val Loss: {best_val:.6f}")

except KeyboardInterrupt:
    print(f"\nInterrupted at epoch {epoch}. Saving checkpoint...")
    save_state(epoch, checkpoint_path)
    with open(log_path, 'w') as f:
        json.dump({'epoch': epoch, 'train_losses': train_losses, 'val_losses': val_losses, 'config': config, 'best_val': best_val}, f, indent=2)
    print(f"Checkpoint saved. Resume from epoch {epoch + 1}")

except Exception as e:
    print(f"\nError at epoch {epoch}: {e}. Saving checkpoint...")
    try:
        save_state(epoch, checkpoint_path)
        with open(log_path, 'w') as f:
            json.dump({'epoch': epoch, 'train_losses': train_losses, 'val_losses': val_losses, 'config': config, 'best_val': best_val}, f, indent=2)
        print(f"Checkpoint saved. Resume from epoch {epoch + 1}")
    except Exception:
        pass
    raise


Starting fresh training run
Epoch    0 | Train Loss: 0.790027 | Val Loss: 0.608500 | Best: 0.608500
Epoch    1 | Train Loss: 0.547045
Epoch    2 | Train Loss: 0.398433
Epoch    3 | Train Loss: 0.296036
Epoch    4 | Train Loss: 0.257464
Epoch    5 | Train Loss: 0.199039
Epoch    6 | Train Loss: 0.183908
Epoch    7 | Train Loss: 0.148751
Epoch    8 | Train Loss: 0.124496
Epoch    9 | Train Loss: 0.130753
Epoch   10 | Train Loss: 0.142380 | Val Loss: 0.102735 | Best: 0.102735
Epoch   11 | Train Loss: 0.140253
Epoch   12 | Train Loss: 0.125902
Epoch   13 | Train Loss: 0.135175
Epoch   14 | Train Loss: 0.107517
Epoch   15 | Train Loss: 0.109996
Epoch   16 | Train Loss: 0.113657
Epoch   17 | Train Loss: 0.107231
Epoch   18 | Train Loss: 0.116387
Epoch   19 | Train Loss: 0.104426
Epoch   20 | Train Loss: 0.093464 | Val Loss: 0.063412 | Best: 0.063412
Epoch   21 | Train Loss: 0.108779
Epoch   22 | Train Loss: 0.085612
Epoch   23 | Train Loss: 0.088556
Epoch   24 | Train Loss: 0.112454
Epoch  

## GT vs Predicted from Forward Noise
This evaluation diffuses one sample with known Gaussian noise `q(x_t | x_0)`, then reconstructs `x_0` from the model prediction and compares GT vs predicted geometry.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('plotly') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly'])

import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'vscode' if 'vscode' in pio.renderers else 'notebook_connected'

def chamfer_l2(a_xyz, b_xyz):
    d = torch.cdist(a_xyz.unsqueeze(0), b_xyz.unsqueeze(0), p=2).squeeze(0)
    return d.min(dim=1).values.mean() + d.min(dim=0).values.mean()

model.eval()
with torch.no_grad():
    eval_loader = val_loader if len(val_loader) > 0 else train_loader
    batch = next(iter(eval_loader))
    x0 = batch['train_points'].transpose(1, 2).to(device)[:1]
    t = torch.full((x0.shape[0],), 500, device=device, dtype=torch.long)
    noise = torch.randn_like(x0)

    x_t = model.diffusion.q_sample(x_start=x0, t=t, noise=noise)
    eps_pred = model._denoise(x_t, t)
    x0_pred = model.diffusion._predict_xstart_from_eps(x_t, t, eps_pred)

    gt = x0[0].transpose(0, 1).contiguous()
    pred = x0_pred[0].transpose(0, 1).contiguous()

    eps_mse = ((eps_pred - noise) ** 2).mean().item()
    x0_mse = ((x0_pred - x0) ** 2).mean().item()
    cd = chamfer_l2(gt, pred).item()

print(f'Eval check | eps MSE: {eps_mse:.6f} | x0 MSE: {x0_mse:.6f} | Chamfer-L2: {cd:.6f}')

gt_np = gt.detach().cpu().numpy()
pred_np = pred.detach().cpu().numpy()

fig_gt = go.Figure()
fig_gt.add_trace(go.Scatter3d(
    x=gt_np[:, 0], y=gt_np[:, 1], z=gt_np[:, 2],
    mode='markers',
    marker=dict(size=3, color='royalblue'),
    name='GT (x0)'
))
fig_gt.update_layout(
    title='GT (x0)',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=650,
    showlegend=False,
    scene_aspectmode='data',
)

fig_pred = go.Figure()
fig_pred.add_trace(go.Scatter3d(
    x=pred_np[:, 0], y=pred_np[:, 1], z=pred_np[:, 2],
    mode='markers',
    marker=dict(size=3, color='tomato'),
    name='Predicted x0 from forward noise'
))
fig_pred.update_layout(
    title=f'Predicted x0 | x0 MSE={x0_mse:.4e}, Chamfer-L2={cd:.4e}',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=650,
    showlegend=False,
    scene_aspectmode='data',
)

fig_gt.show()
fig_pred.show()

Overfit check | eps MSE: 0.004287 | x0 MSE: 0.000502 | Chamfer-L2: 0.029789


In [ ]:
import matplotlib.pyplot as plt

if len(train_losses) == 0:
    print('No training history available yet. Run the training cell first.')
else:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    ax[0].plot(train_losses, label='Train Loss', linewidth=2)
    if len(val_losses) > 0:
        val_epochs = list(range(0, len(train_losses), VAL_INTERVAL))[:len(val_losses)]
        ax[0].plot(val_epochs, val_losses, '--', label='Val Loss', linewidth=2)
    ax[0].set_title('Diffusion Training Loss')
    ax[0].set_xlabel('Epoch')
    ax[0].grid(True, alpha=0.3)
    ax[0].legend()

    if len(train_losses) > 1:
        ax[1].plot(train_losses, linewidth=2)
        ax[1].set_yscale('log')
    else:
        ax[1].plot(train_losses, linewidth=2)
    ax[1].set_title('Train Loss (log scale)')
    ax[1].set_xlabel('Epoch')
    ax[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print('\n=== FINAL STATUS ===')
    print(f'Run: {RUN_NAME}')
    print(f'Final train loss: {train_losses[-1]:.6f}')
    if len(val_losses) > 0:
        print(f'Final val loss: {val_losses[-1]:.6f}')
    print(f'Best val loss: {best_val:.6f}')
    print(f'Checkpoint dir: {checkpoint_dir}')